In [1]:
#Import the libraries 
from __future__ import annotations
 
import argparse
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
 
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline

In [2]:
# 1. CONFIGURATION

DATA_FILE = Path("Study chart (Orignal) - 2.)Study chart.csv")
REQUIRED_COLUMNS = ["Subject", "Time started", "Time ending", "Breaks", "Rating", "Semester"]
TIME_FORMATS = ["%I:%M:%S %p", "%I:%M %p", "%H:%M:%S", "%H:%M"]
 
INTENT_CONFIDENCE = 0.35      # below this the bot switches to open-ended RAG mode
TOP_K = 5                     # records retrieved per question
LLM_MODEL = os.getenv("STUDYBOT_MODEL", "claude-sonnet-5-5")

In [5]:
# 2. LOAD AND PREPROCESS DATAdef load_data(file_path):

def parse_times(series: pd.Series) -> pd.Series:
    """Parse clock times, trying several common formats."""
    parsed = pd.to_datetime(series, format=TIME_FORMATS[0], errors="coerce")
    for fmt in TIME_FORMATS[1:]:
        parsed = parsed.fillna(pd.to_datetime(series, format=fmt, errors="coerce"))
    return parsed
 
 
def normalize_semester(value) -> object:
    """'Semester 2', 'sem2', 'SEM 2', '2'  ->  'SEM 2'."""
    if pd.isna(value):
        return pd.NA
    match = re.search(r"(\d+)", str(value))
    return f"SEM {match.group(1)}" if match else str(value).strip().upper()
 
 
def load_data(file_path) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
 
    missing = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
 
    # FIX: in v1 this line sat after `raise`, so it never ran.
    df = df[REQUIRED_COLUMNS].copy()
 
    # Clean text fields; blank strings become missing values
    for col in REQUIRED_COLUMNS:
        s = df[col].astype("string").str.strip()
        df[col] = s.where(s != "", pd.NA)
 
    df["semester"] = df["Semester"].map(normalize_semester)
    df["rating_clean"] = df["Rating"].str.lower()
 
    df["start_time"] = parse_times(df["Time started"])
    # FIX: v1 parsed "Time started" for the end time too, so every duration was 0.
    df["end_time"] = parse_times(df["Time ending"])
 
    df["duration_hours"] = (df["end_time"] - df["start_time"]).dt.total_seconds() / 3600
    df.loc[df["duration_hours"] < 0, "duration_hours"] += 24          # crossed midnight
    # Keep only plausible durations (0 < h <= 24); everything else becomes NaN
    df["duration_hours"] = df["duration_hours"].where(df["duration_hours"].between(0, 24, inclusive="right"))
 
    df["break_count"] = pd.to_numeric(df["Breaks"].str.extract(r"(\d+)", expand=False), errors="coerce")
    no_breaks = df["Breaks"].str.lower().str.contains(r"\bnone\b|\bno breaks?\b|^no$|^nil$", regex=True, na=False)
    df.loc[no_breaks, "break_count"] = 0
 
    df["session_id"] = np.arange(1, len(df) + 1)
    return df
 
 
def make_demo_data(path: Path, n: int = 120, seed: int = 7) -> Path:
    """Write a realistic sample CSV so the bot can be tried without real data."""
    rng = random.Random(seed)
    subjects = ["Finance", "Accounting", "Statistics", "Marketing", "Economics", "Finance & Accounting"]
    ratings = ["Very productive", "Productive", "Average", "Unproductive"]
    breaks = ["None taken", "1 break", "2 breaks", "3 breaks"]
    rows = []
    for _ in range(n):
        start_h, start_m = rng.randint(6, 21), rng.choice([0, 15, 30, 45])
        length = rng.choice([30, 45, 60, 90, 120, 150, 180, 240])
        end = start_h * 60 + start_m + length
        fmt = lambda mins: pd.Timestamp("2000-01-01") + pd.Timedelta(minutes=mins % 1440)
        rows.append({
            "Subject": rng.choice(subjects),
            "Time started": fmt(start_h * 60 + start_m).strftime("%I:%M:%S %p"),
            "Time ending": fmt(end).strftime("%I:%M:%S %p"),
            "Breaks": rng.choice(breaks),
            "Rating": rng.choices(ratings, weights=[2, 4, 3, 1])[0],
            "Semester": rng.choice(["Sem 1", "Sem 2", "Semester 3", "SEM 4"]),
        })
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

In [6]:
#NLP TRAINING DATA 
training_data={
    "subject":[
        "Which subject did I study the most?",
        "Show my most studied subject",
        "Count my sessions by subject",
        "How many sessions did I have for each subject?",
        "Which subject did I study?",
        "How often did I study finance?",
        "What is my most frequent subject?",
        "Show my subject statistics"
    ],

    "productivty":[
        "How productive was I?",
        "Show my productivity ratings",
        "Show the distribution of ratings",
        "How many sessions were rated productive?",
        "What was my most common rating?",
        "How many highly rated sessions did I have?",
        "Show my study performance ratings",
        "What ratings did I receive?"
    ],

    "duration":[
        "How productive was I?",
        "Show my productivity ratings",
        "Show the distribution of ratings",
        "How many sessions were rated productive?",
        "What was my most common rating?",
        "How many highly rated sessions did I have?",
        "Show my study performance ratings",
        "What ratings did I receive?"
    ],

    
    "breaks": [
        "How many breaks did I take?",
        "Show my average number of breaks",
        "How many sessions had no breaks?",
        "What is my break pattern?",
        "How often did I take breaks?",
        "Show my break statistics",
        "What was the average number of breaks?",
        "Calculate my breaks"
    ],

    "semester": [
        "Show my study sessions by semester",
        "How did I perform in semester 1?",
        "Compare my semesters",
        "Which semester had the most sessions?",
        "Show the ratings for semester 3",
        "How many sessions were in each semester?",
        "Compare my semester performance",
        "Show my semester statistics"
    ],

    "help": [
        "Help me use the chatbot",
        "What can you do?",
        "Show available commands",
        "What questions can I ask?",
        "How does this chatbot work?",
        "Show me some examples"
    ]
}

In [8]:
# 3. INTENT CLASSIFIER  (hand-written + synthetic training data)

TRAINING_DATA = {
    "subject": [
        "Which subject did I study the most?", "Show my most studied subject",
        "Count my sessions by subject", "How many sessions did I have for each subject?",
        "Which subject did I study?", "How often did I study finance?",
        "What is my most frequent subject?", "Show my subject statistics",
    ],
    "productivity": [
        "How productive was I?", "Show my productivity ratings", "Show the distribution of ratings",
        "How many sessions were rated productive?", "What was my most common rating?",
        "How many highly rated sessions did I have?", "Show my study performance ratings",
        "What ratings did I receive?", "Show the ratings for semester 3",
    ],
    "duration": [
        "How many hours did I study?", "What was my longest study session?",
        "Show my average session length", "What is my total study time?",
        "What was my shortest session?", "How long do I usually study?",
        "Show my session duration statistics", "How much time did I spend studying?",
    ],
    "breaks": [
        "How many breaks did I take?", "Show my average number of breaks",
        "How many sessions had no breaks?", "What is my break pattern?",
        "How often did I take breaks?", "Show my break statistics",
        "What was the average number of breaks?", "Calculate my breaks",
    ],
    "semester": [
        "Show my study sessions by semester", "Compare my semesters",
        "Which semester had the most sessions?", "How many sessions were in each semester?",
        "Compare my semester performance", "Show my semester statistics",
    ],
    "overview": [
        "Give me an overview", "Summarize my study habits", "How did I perform in semester 1?",
        "Overall study summary", "Tell me about my study patterns", "How am I doing overall?",
    ],
    "help": [
        "Help me use the chatbot", "What can you do?", "Show available commands",
        "What questions can I ask?", "How does this chatbot work?", "Show me some examples",
    ],
}
 
# Building blocks used to SYNTHESIZE many more training questions
_CORE_PHRASES = {
    "subject": ["which subject did I study the most", "my most studied subject", "number of sessions per subject",
                "subject breakdown", "how many sessions for each subject", "which subject did I study least"],
    "productivity": ["how productive was I", "my productivity ratings", "rating distribution",
                     "how many sessions were rated productive", "what was my most common rating",
                     "how did I rate my sessions"],
    "duration": ["how many hours did I study", "my longest study session", "my shortest session",
                 "average session length", "total study time", "how long do I usually study"],
    "breaks": ["how many breaks did I take", "my average number of breaks", "how many sessions had no breaks",
               "break statistics", "how often do I take breaks"],
    "semester": ["compare my semesters", "sessions by semester", "which semester had the most sessions",
                 "semester statistics", "how many sessions in each semester"],
    "overview": ["give me an overview", "summarize my study habits", "overall study summary",
                 "tell me about my study patterns", "what are my study insights", "how am I doing overall"],
}
_OPENERS = ["", "please ", "can you ", "could you tell me ", "i want to know ", "tell me ", "quick question: "]
_SCOPES = ["", " in semester 1", " for sem 2", " during semester 3", " in the first semester"]
 
 
def synthesize_training_data(subjects, n_per_intent: int = 40, seed: int = 42) -> dict:
    """Generate extra questions by mixing openers x core phrases x scopes (incl. real subject names)."""
    rng = random.Random(seed)
    scopes = _SCOPES + [f" for {s.lower()}" for s in list(subjects)[:8]]
    out = {}
    for intent, cores in _CORE_PHRASES.items():
        samples, tries = set(), 0
        while len(samples) < n_per_intent and tries < n_per_intent * 20:
            tries += 1
            text = f"{rng.choice(_OPENERS)}{rng.choice(cores)}{rng.choice(scopes)}".strip()
            samples.add(text)
        out[intent] = sorted(samples)
    return out
 
 
def train_model(subjects=()) -> Pipeline:
    data = {k: list(v) for k, v in TRAINING_DATA.items()}
    for intent, extra in synthesize_training_data(subjects).items():
        data[intent].extend(extra)
 
    questions, labels = [], []
    for intent, examples in data.items():
        questions.extend(examples)
        labels.extend([intent] * len(examples))
 
    model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)),
        ("classifier", LogisticRegression(C=10, max_iter=2000)),
    ])
    model.fit(questions, labels)
    return model
 
 
# ---------------------------------------------------------------------------
# 4. ENTITY EXTRACTION
# ---------------------------------------------------------------------------
_ORDINALS = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5, "sixth": 6, "seventh": 7, "eighth": 8}
 
 
def extract_semester(question: str) -> Optional[str]:
    # FIX: v1 searched for upper-case 'SEM (\d+)' in the raw question, so
    # "semester 1" / "sem 2" typed in lower-case never matched.
    q = question.lower()
    m = re.search(r"\bsem(?:ester)?\s*[-#:]?\s*(\d+)\b", q)
    if m:
        return f"SEM {m.group(1)}"
    m = re.search(r"\b(" + "|".join(_ORDINALS) + r")\s+sem(?:ester)?\b", q)
    if m:
        return f"SEM {_ORDINALS[m.group(1)]}"
    return None
 
 
def extract_subject(question: str, df: pd.DataFrame) -> Optional[str]:
    q = question.lower()
    subjects = df["Subject"].dropna().astype(str).unique()
    for subject in sorted(subjects, key=len, reverse=True):      # longest label first
        label = subject.strip().lower()
        if label and re.search(r"(?<!\w)" + re.escape(label) + r"(?!\w)", q):
            return subject
    return None
 
 
def filter_data(df: pd.DataFrame, semester=None, subject=None) -> pd.DataFrame:
    data = df
    if semester:
        data = data[(data["semester"] == semester).fillna(False).to_numpy(dtype=bool)]
    if subject:
        data = data[data["Subject"].str.lower().eq(subject.lower()).fillna(False).to_numpy(dtype=bool)]
    return data

In [9]:
# 5. ANALYTICS LAYER (exact numbers)


def extract_semester(question):
    match = re.search(r'SEM (\d+)', question)  # Corrected regex pattern

    if match:
        return f"SEM {match.group(1)}"

    return None

def extract_subject(question, df):
    question = question.lower()

    subjects = (
        df["Subject"]
        .dropna()
        .astype(str)
        .unique()
    )

    # Check longer subject names first
    for subject in sorted(subjects, key=len, reverse=True):
        subject_clean = subject.strip().lower()

        if subject_clean and subject_clean in question:
            return subject

    return None

In [10]:
# 5. ANALYTICS LAYER (exact numbers)

def compute_stats(data: pd.DataFrame) -> dict:
    dur = data["duration_hours"].dropna()
    brk = data["break_count"].dropna()
    rat = data["rating_clean"].dropna()
    rating_counts = rat.value_counts()
    return {
        "sessions": int(len(data)),
        "valid_durations": int(len(dur)),
        "total_hours": float(dur.sum()) if len(dur) else 0.0,
        "avg_hours": float(dur.mean()) if len(dur) else None,
        "longest": float(dur.max()) if len(dur) else None,
        "shortest": float(dur.min()) if len(dur) else None,
        "break_records": int(len(brk)),
        "avg_breaks": float(brk.mean()) if len(brk) else None,
        "no_break_sessions": int((brk == 0).sum()),
        "ratings": {str(k): int(v) for k, v in rating_counts.items()},
        "rated": int(len(rat)),
    }
 
 
def _scope(subject, semester) -> str:
    parts = [p for p in (subject, semester) if p]
    return f" ({', '.join(parts)})" if parts else ""
 
 
def _ratings_lines(st: dict) -> str:
    total = st["rated"]
    return "\n".join(f"- {r.title()}: {c} ({c / total:.1%})" for r, c in st["ratings"].items())
 
 
def describe_stats(label: str, st: dict) -> str:
    """One-paragraph summary used for summary documents and as general context."""
    bits = [f"{label}: {st['sessions']} study sessions"]
    if st["valid_durations"]:
        bits.append(f"{st['total_hours']:.1f} total hours, average session {st['avg_hours']:.2f} hours")
    if st["avg_breaks"] is not None:
        bits.append(f"average {st['avg_breaks']:.2f} breaks per session ({st['no_break_sessions']} with no breaks)")
    if st["ratings"]:
        bits.append("ratings: " + ", ".join(f"{r} {c}" for r, c in st["ratings"].items()))
    return "; ".join(bits) + "."
 
 
def analyze_subjects(data, q, subject, semester) -> str:
    scope = f" in {semester}" if semester else ""
    if subject:
        st = compute_stats(data)
        return f"{subject}{scope}: {st['sessions']} recorded sessions, {st['total_hours']:.1f} hours in total."
 
    counts = data["Subject"].dropna().value_counts()
    if counts.empty:
        return "No subject data is available."
 
    if re.search(r"\b(least|fewest|lowest)\b", q.lower()):
        msg = f"Least studied subject{scope}: {counts.index[-1]} ({counts.iloc[-1]} sessions)."
    else:
        msg = f"Most studied subject{scope} (by sessions): {counts.index[0]} ({counts.iloc[0]} sessions)."
        hours = data.groupby("Subject")["duration_hours"].sum()
        hours = hours[hours > 0].sort_values(ascending=False)
        if not hours.empty and hours.index[0] != counts.index[0]:
            msg += f" By total hours it is {hours.index[0]} ({hours.iloc[0]:.1f} h)."
 
    lines = "\n".join(f"- {n}: {c} sessions" for n, c in counts.head(10).items())
    return f"{msg}\nSessions per subject:\n{lines}\nNote: combined-subject labels are counted separately."
 
 
def analyze_productivity(data, q, subject, semester) -> str:
    st = compute_stats(data)
    if not st["ratings"]:
        return "No rating data is available."
    top, top_n = next(iter(st["ratings"].items()))
    return (f"Study rating distribution{_scope(subject, semester)}:\n{_ratings_lines(st)}\n"
            f"Most common rating: {top.title()} ({top_n}). Rated sessions: {st['rated']}.")
 
 
def analyze_durations(data, q, subject, semester) -> str:
    st = compute_stats(data)
    if not st["valid_durations"]:
        return "No valid session durations were found."
    ql = q.lower()
    if "longest" in ql:
        return f"Longest session{_scope(subject, semester)}: {st['longest']:.2f} hours."
    if "shortest" in ql:
        return f"Shortest session{_scope(subject, semester)}: {st['shortest']:.2f} hours."
    if "total" in ql or "overall" in ql or "how many hours" in ql or "how much time" in ql:
        return (f"Total elapsed study time{_scope(subject, semester)}: {st['total_hours']:.2f} hours "
                f"across {st['valid_durations']} valid sessions.")
    return (f"Session durations{_scope(subject, semester)}:\n"
            f"- Average: {st['avg_hours']:.2f} h\n- Total: {st['total_hours']:.2f} h\n"
            f"- Longest: {st['longest']:.2f} h\n- Shortest: {st['shortest']:.2f} h\n"
            f"- Valid sessions: {st['valid_durations']}")
 
 
def analyze_breaks(data, q, subject, semester) -> str:
    st = compute_stats(data)
    if st["avg_breaks"] is None:
        return "No usable break-count data was found."
    return (f"Breaks{_scope(subject, semester)}:\n"
            f"- Average breaks per session: {st['avg_breaks']:.2f}\n"
            f"- Sessions with no breaks: {st['no_break_sessions']}\n"
            f"- Sessions with break data: {st['break_records']}")
 
 
def _sem_key(label: str) -> int:
    m = re.search(r"\d+", label)
    return int(m.group()) if m else 999
 
 
def analyze_semesters(data, q, subject, semester) -> str:
    if semester:
        return describe_stats(f"{semester}{_scope(subject, None)}", compute_stats(data))
    labels = sorted(data["semester"].dropna().unique(), key=_sem_key)
    if not labels:
        return "No semester information is available."
    lines = []
    for label in labels:
        st = compute_stats(data[data["semester"] == label])
        top = next(iter(st["ratings"]), "n/a").title()
        lines.append(f"- {label}: {st['sessions']} sessions, {st['total_hours']:.1f} h, top rating: {top}")
    best = max(labels, key=lambda s: int((data["semester"] == s).sum()))
    return f"Study records by semester{_scope(subject, None)}:\n" + "\n".join(lines) + f"\nMost sessions: {best}."
 
 
def analyze_overview(data, q, subject, semester) -> str:
    return describe_stats(f"Overview{_scope(subject, semester)}", compute_stats(data))
 
 
ANALYZERS = {
    "subject": analyze_subjects,
    "productivity": analyze_productivity,
    "duration": analyze_durations,
    "breaks": analyze_breaks,
    "semester": analyze_semesters,
    "overview": analyze_overview,
}
 
 
def generate_insights(st: dict) -> list:
    """Small rule-based observations that get woven into the answer."""
    notes = []
    if st["ratings"] and st["rated"]:
        top, n = next(iter(st["ratings"].items()))
        if n / st["rated"] >= 0.4:
            notes.append(f"{n / st['rated']:.0%} of rated sessions were '{top}'.")
    if st["break_records"] and st["no_break_sessions"] / st["break_records"] > 0.5:
        notes.append("More than half of your sessions had no breaks.")
    if st["avg_hours"] and st["avg_hours"] >= 2.5:
        notes.append(f"Your typical session is long ({st['avg_hours']:.1f} h on average).")
    return notes

In [11]:
# 6. RAG: KNOWLEDGE BASE + RETRIEVER 
@dataclass
class Doc:
    text: str
    kind: str                       # "session" | "summary"
    semester: Optional[str] = None
    subject: Optional[str] = None
 
 
def _semester_phrase(sem) -> str:
    if sem is None or pd.isna(sem):
        return "unknown semester"
    m = re.search(r"\d+", str(sem))
    return f"semester {m.group()} ({sem})" if m else str(sem)
 
 
def _clock(ts) -> str:
    return ts.strftime("%I:%M %p").lstrip("0") if pd.notna(ts) else "unknown time"
 
 
def build_knowledge_base(df: pd.DataFrame) -> list:
    """Turn the table into text documents: one per session + aggregate summaries."""
    docs = []
    for row in df.itertuples():
        subject = None if pd.isna(row.Subject) else str(row.Subject)
        sem = None if pd.isna(row.semester) else str(row.semester)
        dur = f"{row.duration_hours:.2f} hours" if pd.notna(row.duration_hours) else "duration unknown"
        breaks = "unknown" if pd.isna(row.Breaks) else row.Breaks
        rating = "not rated" if pd.isna(row.Rating) else row.Rating
        text = (f"Session {row.session_id}: {subject or 'unknown subject'} in {_semester_phrase(sem)}, "
                f"{_clock(row.start_time)} to {_clock(row.end_time)} ({dur}), breaks: {breaks}, rating: {rating}.")
        docs.append(Doc(text, "session", sem, subject))
 
    docs.append(Doc(describe_stats("Overall study summary", compute_stats(df)), "summary"))
    for subject, grp in df.groupby("Subject"):
        docs.append(Doc(describe_stats(f"Summary for subject {subject}", compute_stats(grp)),
                        "summary", None, str(subject)))
    for sem, grp in df.groupby("semester"):
        docs.append(Doc(describe_stats(f"Summary for {_semester_phrase(sem)}", compute_stats(grp)),
                        "summary", str(sem), None))
    return docs
 
 
class Retriever:
    """TF-IDF retriever with metadata filtering (semester / subject)."""
 
    def __init__(self, docs: list):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
        self.matrix = self.vectorizer.fit_transform([d.text for d in docs])
 
    @staticmethod
    def _matches(doc: Doc, semester, subject) -> bool:
        return ((semester is None or doc.semester == semester)
                and (subject is None or doc.subject == subject))
 
    def search(self, query: str, k: int = TOP_K, semester=None, subject=None) -> list:
        scores = cosine_similarity(self.vectorizer.transform([query]), self.matrix).ravel()
        filtered = bool(semester or subject)
        mask = np.array([self._matches(d, semester, subject) for d in self.docs])
        if filtered and mask.any():
            scores = np.where(mask, scores, -1.0)
        order = np.argsort(-scores)[:k]
        hits = [self.docs[i] for i in order if scores[i] > 0]
        if not hits and filtered and mask.any():          # no lexical overlap: fall back to the slice itself
            idx = [i for i in np.flatnonzero(mask)]
            idx.sort(key=lambda i: (self.docs[i].kind != "summary", i))
            hits = [self.docs[i] for i in idx[:k]]
        return hits

In [121]:
# 7. SYNTHESIZER (grounded answer generation) 
SYSTEM_PROMPT = (
    "You are StudyBot, a friendly assistant that answers questions about a student's study log. "
    "Answer ONLY from the 'Computed facts' and 'Retrieved records' provided. Computed facts are exact "
    "and must never be contradicted. If the data does not contain the answer, say so plainly. "
    "Be concise (under 120 words), plain text, no headings."
)
 
 
class Synthesizer:
    def __init__(self, use_llm: bool = True):
        self.client = None
        if use_llm and os.getenv("ANTHROPIC_API_KEY"):
            try:
                import anthropic
                self.client = anthropic.Anthropic()
            except ImportError:
                print("(anthropic package not installed; using offline synthesis)")
 
    @property
    def mode(self) -> str:
        return f"LLM ({LLM_MODEL})" if self.client else "offline"
 
    def synthesize(self, question, facts, evidence, insights, history) -> str:
        if self.client:
            try:
                return self._llm(question, facts, evidence, history)
            except Exception as err:                       # network/key/model problems -> degrade gracefully
                print(f"[LLM unavailable ({type(err).__name__}); switching to offline synthesis]")
                self.client = None
        return self._offline(facts, evidence, insights)
 
    def _llm(self, question, facts, evidence, history) -> str:
        records = "\n".join(f"- {d.text}" for d in evidence) or "(none)"
        turns = "\n".join(f"User: {q}\nStudyBot: {a}" for q, a in history[-3:]) or "(none)"
        prompt = (f"Recent conversation:\n{turns}\n\nQuestion: {question}\n\n"
                  f"Computed facts:\n{facts or '(none)'}\n\nRetrieved records:\n{records}")
        resp = self.client.messages.create(
            model=LLM_MODEL, max_tokens=500, system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text").strip()
 
    @staticmethod
    def _offline(facts, evidence, insights) -> str:
        parts = [facts] if facts else []
        if insights:
            parts.append("What stands out:\n" + "\n".join(f"- {i}" for i in insights))
        sessions = [d for d in evidence if d.kind == "session"][:3]
        if sessions:
            parts.append("Supporting records:\n" + "\n".join(f"- {d.text}" for d in sessions))
        elif not facts and evidence:
            parts.append("\n".join(f"- {d.text}" for d in evidence[:3]))
        return "\n\n".join(parts) if parts else "I couldn't find anything relevant in your study log."

In [12]:
# 8. THE CHATBOT 
HELP_TEXT = (
    "I can answer questions about your study log. Try:\n"
    "- Which subject did I study the most?\n"
    "- How many sessions were rated productive in semester 2?\n"
    "- What was my longest study session?\n"
    "- What is my average number of breaks for finance?\n"
    "- Compare my semesters / Give me an overview\n"
    "- Or ask anything open-ended, e.g. 'When did I last study statistics?'\n"
    "Type 'exit' to quit."
)
_SMALL_TALK = re.compile(r"^\s*(hi|hello|hey|good (morning|afternoon|evening)|thanks|thank you)\b", re.I)
 
 
class StudyBot:
    def __init__(self, df: pd.DataFrame, use_llm: bool = True):
        self.df = df
        self.model = train_model(df["Subject"].dropna().unique())
        self.retriever = Retriever(build_knowledge_base(df))
        self.synth = Synthesizer(use_llm)
        self.history: list = []
        self.last_intent: Optional[str] = None
 
    @classmethod
    def from_csv(cls, path, use_llm: bool = True) -> "StudyBot":
        return cls(load_data(path), use_llm)
 
    def classify(self, question: str):
        probs = self.model.predict_proba([question])[0]
        i = int(np.argmax(probs))
        return self.model.classes_[i], float(probs[i])
 
    def ask(self, question: str) -> str:
        question = question.strip()
        if not question:
            return "Please enter a question."
        if question.lower().strip(" ?!.") in {"help", "commands", "examples", "menu"}:
            return HELP_TEXT
        if _SMALL_TALK.match(question):
            return "Hello! Ask me about your study sessions, or type 'help' for examples."
 
        intent, conf = self.classify(question)
        semester = extract_semester(question)
        subject = extract_subject(question, self.df)
 
        # Follow-ups such as "what about semester 2?" reuse the previous intent
        if (conf < INTENT_CONFIDENCE and (semester or subject)
                and self.last_intent and len(question.split()) <= 6):
            intent, conf = self.last_intent, INTENT_CONFIDENCE
 
        confident = conf >= INTENT_CONFIDENCE
        if confident and intent == "help":
            return HELP_TEXT
 
        data = filter_data(self.df, semester, subject)
        if data.empty:
            return f"No records found for {', '.join(p for p in (subject, semester) if p)}."
 
        if confident:
            facts = ANALYZERS[intent](data, question, subject, semester)
            insights = generate_insights(compute_stats(data)) if intent == "overview" else []
            self.last_intent = intent
        else:  # open-ended question -> RAG-only mode with general context
            facts = describe_stats(f"Study log{_scope(subject, semester)}", compute_stats(data))
            insights = []
 
        evidence = self.retriever.search(question, semester=semester, subject=subject)
        answer = self.synth.synthesize(question, facts, evidence, insights, self.history)
        self.history.append((question, answer))
        return answer

In [13]:
# 9. COMMAND-LINE INTERFACE
def main(argv=None) -> None:
    parser = argparse.ArgumentParser(description="StudyBot - RAG study assistant")
    parser.add_argument("--data", type=Path, default=DATA_FILE, help="path to the Study chart CSV")
    parser.add_argument("--demo", action="store_true", help="generate and use sample data")
    parser.add_argument("--no-llm", action="store_true", help="disable LLM synthesis")
    args, _ = parser.parse_known_args(argv)        # tolerant of Jupyter's extra arguments
 
    data_path = make_demo_data(Path("demo_study_chart.csv")) if args.demo else args.data
    try:
        bot = StudyBot.from_csv(data_path, use_llm=not args.no_llm)
    except FileNotFoundError:
        print(f"Dataset not found: {data_path}\nPass --data <file.csv> or try --demo.")
        return
    except (ValueError, pd.errors.ParserError) as err:
        print(f"Could not load the dataset: {err}")
        return
 
    print("=" * 52)
    print("      STUDYBOT: RAG STUDY ASSISTANT")
    print("=" * 52)
    print(f"Loaded {len(bot.df)} study records | synthesis mode: {bot.synth.mode}")
    print("Type 'help' for examples or 'exit' to quit.")
 
    while True:
        try:
            question = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nStudyBot: Goodbye.")
            break
        if question.lower() in {"exit", "quit", "bye"}:
            print("StudyBot: Goodbye.")
            break
        print(f"\nStudyBot: {bot.ask(question)}")
 
 
if __name__ == "__main__":
    main()

Dataset not found: Study chart (Orignal) - 2.)Study chart.csv
Pass --data <file.csv> or try --demo.
